In [ ]:
from google.colab import files
files.upload()

1. Tải lên và giải nén dữ liệu
Cell đầu tiên (tải file zip): Dòng from google.colab import files và files.upload() cho phép bạn tải file dữ liệu (thường là file zip) từ máy tính lên môi trường Google Colab.
Cell thứ hai (giải nén): Lệnh !unzip -o "Garbage classification.zip" dùng để giải nén file zip mà bạn vừa tải lên. -o có nghĩa là ghi đè nếu file đã tồn tại. Sau bước này, bạn sẽ có một thư mục Garbage classification chứa các ảnh rác.

In [ ]:
!unzip -o "Garbage classification.zip"

### 2. Import các thư viện cần thiết

- **`torch`, `torch.nn`, `torch.optim`**: Đây là các thư viện cốt lõi của PyTorch để xây dựng và huấn luyện mô hình mạng nơ-ron.
- **`torchvision.datasets`, `torchvision.transforms`, `torchvision.models`**: Các thư viện của PyTorch dành cho thị giác máy tính. `datasets` dùng để tải các bộ dữ liệu phổ biến, `transforms` để tiền xử lý ảnh (thay đổi kích thước, chuyển đổi sang tensor, v.v.), và `models` cung cấp các mô hình mạng nơ-ron đã được huấn luyện trước (pretrained models).
- **`torch.utils.data.DataLoader`**: Dùng để tạo các đối tượng DataLoader, giúp nạp dữ liệu theo lô (batch) một cách hiệu quả trong quá trình huấn luyện.
- **`matplotlib.pyplot`**: Thư viện dùng để vẽ biểu đồ và hiển thị hình ảnh.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

### 3. Chuẩn bị dữ liệu và chia tập huấn luyện/kiểm tra

- **`data_dir`**: Định nghĩa đường dẫn đến thư mục chứa ảnh đã giải nén.
- **`transform`**: Định nghĩa các bước tiền xử lý ảnh:
    - `transforms.Resize((224, 224))`: Thay đổi kích thước tất cả ảnh về 224x224 pixel.
    - `transforms.ToTensor()`: Chuyển ảnh từ định dạng PIL Image hoặc NumPy array sang PyTorch Tensor và tự động scale các giá trị pixel về khoảng [0, 1].
- **`dataset = datasets.ImageFolder(...)`**: Tạo một đối tượng dataset từ thư mục ảnh. `ImageFolder` tự động gán nhãn (label) cho ảnh dựa vào tên thư mục con (ví dụ: `cardboard`, `glass`, v.v.).
- **Chia tập dữ liệu (Train/Validation/Test split)**:
    - Dữ liệu được chia thành 3 tập: huấn luyện (70%), kiểm định (15%), và kiểm tra (15%).
    - Việc chia tập được thực hiện *theo lớp* (stratified split) để đảm bảo mỗi tập con có tỷ lệ các lớp rác tương tự nhau, tránh tình trạng một lớp bị thiếu hoặc quá nhiều trong một tập.
    - `train_loader`, `val_loader`, `test_loader`: Tạo các DataLoader để nạp dữ liệu theo batch trong quá trình huấn luyện và đánh giá. `batch_size=32` nghĩa là mỗi lần nạp 32 ảnh.
- **In ra thông tin**: Hiển thị tên các lớp và số lượng ảnh trong mỗi tập con.

In [ ]:
data_dir = "Garbage classification"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# --- Train/Validation/Test split ---
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15
seed = 42

assert abs((train_ratio + val_ratio + test_ratio) - 1.0) < 1e-9

# Chia theo lớp (stratified) để các tập có phân bố lớp tương tự nhau
from collections import defaultdict
import random
from torch.utils.data import Subset

targets = getattr(dataset, "targets", None)
if targets is None:
    targets = [label for _, label in dataset.samples]

indices_by_class = defaultdict(list)
for idx, label in enumerate(targets):
    indices_by_class[int(label)].append(idx)

rng = random.Random(seed)
train_indices, val_indices, test_indices = [], [], []
for label, idxs in indices_by_class.items():
    rng.shuffle(idxs)
    n = len(idxs)
    n_train = int(train_ratio * n)
    n_val = int(val_ratio * n)
    n_test = n - n_train - n_val

    train_indices.extend(idxs[:n_train])
    val_indices.extend(idxs[n_train : n_train + n_val])
    test_indices.extend(idxs[n_train + n_val :])

rng.shuffle(train_indices)
rng.shuffle(val_indices)
rng.shuffle(test_indices)

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)
test_dataset = Subset(dataset, test_indices)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Classes:", dataset.classes)
print(f"Split sizes -> Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

### 4. Tải và cấu hình mô hình ResNet18

- **Tải ResNet18 pretrained**: Tải mô hình ResNet18 đã được huấn luyện trên tập dữ liệu ImageNet (`pretrained=True` hoặc `weights=models.ResNet18_Weights.DEFAULT`). Việc sử dụng mô hình pretrained giúp mô hình hội tụ nhanh hơn và đạt hiệu suất tốt hơn với ít dữ liệu hơn (kỹ thuật Transfer Learning).
- **Thay đổi lớp đầu ra (Fully Connected Layer)**:
    - `model.fc = nn.Linear(model.fc.in_features, num_classes)`: ResNet18 gốc được thiết kế cho 1000 lớp của ImageNet. Ở đây, chúng ta thay thế lớp Fully Connected cuối cùng (`model.fc`) bằng một lớp mới có số đầu ra là `num_classes` (số lượng các loại rác, trong trường hợp này là 6). Các trọng số của lớp này sẽ được huấn luyện từ đầu.
- **`device = torch.device(...)`**: Xác định thiết bị sẽ dùng để tính toán (GPU nếu có `cuda`, nếu không thì dùng CPU). Việc này giúp tận dụng sức mạnh của GPU để huấn luyện nhanh hơn.
- **`model = model.to(device)`**: Chuyển mô hình sang thiết bị đã chọn.

In [ ]:
# Tải ResNet18 pretrained (ImageNet) - tương thích nhiều phiên bản torchvision
try:
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
except Exception:
    model = models.resnet18(pretrained=True)

# Số lớp đầu ra (mặc định 6 lớp rác)
num_classes = 6
try:
    num_classes = len(dataset.classes)
except Exception:
    pass

model.fc = nn.Linear(model.fc.in_features, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

### 5. Hàm đánh giá mô hình (`evaluate`)

- Hàm này được dùng để tính toán loss (sai số) và accuracy (độ chính xác) của mô hình trên một tập dữ liệu (thường là Validation hoặc Test set).
- **`model.eval()`**: Đặt mô hình vào chế độ đánh giá. Điều này vô hiệu hóa các lớp như Dropout và Batch Normalization hoạt động khác trong quá trình huấn luyện.
- **`with ctx():`**: `torch.inference_mode` (hoặc `torch.no_grad` cho các phiên bản cũ hơn) được sử dụng để tắt tính toán gradient. Điều này giúp tăng tốc độ đánh giá và giảm mức sử dụng bộ nhớ vì chúng ta không cần backpropagation trong khi đánh giá.
- **Loop qua `loader`**: Lấy từng batch ảnh và nhãn.
- **`outputs = model(images)`**: Đưa ảnh qua mô hình để nhận dự đoán.
- **`loss = criterion(outputs, labels)`**: Tính toán giá trị loss.
- **Tính `total_loss`, `correct`, `total`**: Cộng dồn các giá trị để tính loss trung bình và độ chính xác tổng thể.

In [ ]:
def evaluate(model, loader, criterion, device):
    """Tính loss và accuracy cho validation/test."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    ctx = torch.inference_mode if hasattr(torch, "inference_mode") else torch.no_grad
    with ctx():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / max(1, total)
    acc = correct / max(1, total)
    return avg_loss, acc

### 6. Huấn luyện mô hình

- **`criterion = nn.CrossEntropyLoss()`**: Định nghĩa hàm mất mát (loss function). `CrossEntropyLoss` thường được dùng cho các bài toán phân loại đa lớp.
- **`optimizer = optim.Adam(model.parameters(), lr=0.001)`**: Định nghĩa thuật toán tối ưu hóa. Adam là một thuật toán phổ biến với tốc độ học (`lr`) là 0.001.
- **`epochs = 10`**: Số lần toàn bộ tập huấn luyện được đưa qua mô hình.
- **Vòng lặp huấn luyện**: Mỗi epoch bao gồm:
    - **`model.train()`**: Đặt mô hình vào chế độ huấn luyện.
    - **Vòng lặp qua `train_loader`**: 
        - `optimizer.zero_grad()`: Xóa gradient của các tham số ở lần lặp trước.
        - `outputs = model(images)`: Dự đoán của mô hình.
        - `loss = criterion(outputs, labels)`: Tính toán loss.
        - `loss.backward()`: Tính toán gradient của loss theo các tham số của mô hình (backpropagation).
        - `optimizer.step()`: Cập nhật trọng số của mô hình dựa trên gradient.
    - **Đánh giá trên tập Validation**: Gọi hàm `evaluate` trên `val_loader` để xem hiệu suất của mô hình trên dữ liệu chưa thấy.
    - **Lưu mô hình tốt nhất**: Nếu độ chính xác trên tập validation cải thiện, lưu lại trạng thái (trọng số) của mô hình.
- **Kết quả huấn luyện**: In ra loss và accuracy cho tập huấn luyện và validation sau mỗi epoch.
- **Tải lại mô hình tốt nhất**: Sau khi huấn luyện xong, mô hình sẽ tải lại trọng số đã cho kết quả tốt nhất trên tập validation.
- **Đánh giá trên tập Test**: Cuối cùng, mô hình được đánh giá trên tập kiểm tra (`test_loader`) để có được ước lượng hiệu suất không thiên vị của mô hình.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

import copy
best_val_acc = -1.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(epochs):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / max(1, train_total)
    train_acc = train_correct / max(1, train_total)

    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train loss: {train_loss:.4f} | Train acc: {train_acc*100:.2f}% | "
        f"Val loss: {val_loss:.4f} | Val acc: {val_acc*100:.2f}%"
    )

# Load lại mô hình tốt nhất theo Validation trước khi đánh giá Test / lưu model
model.load_state_dict(best_model_wts)
print(f"Best Val acc: {best_val_acc*100:.2f}%")

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test loss: {test_loss:.4f} | Test acc: {test_acc*100:.2f}%")

### 7. Lưu và tải mô hình đã huấn luyện

- **`torch.save(model.state_dict(), "garbage_model.pth")`**: Lưu trạng thái (state_dict) của mô hình tốt nhất vào một file có tên `garbage_model.pth`. `state_dict()` chỉ lưu các tham số đã học của mô hình, không phải toàn bộ cấu trúc mô hình.
- **`files.download("garbage_model.pth")`**: Dòng code này sử dụng `google.colab.files` để tải file mô hình `garbage_model.pth` về máy tính của bạn.

In [ ]:
torch.save(model.state_dict(), "garbage_model.pth")
from google.colab import files
files.download("garbage_model.pth")